# 03 — Reproduce the published NOWRITE result

**Stage:** Gautam's step 2, and the proposal's **Week 6 milestone**: *"Reproducing the
published 38–42% changed-answer rate under NOWRITE on the 3B checkpoints is the cleanest
available evidence that our instrumented fork behaves like the real system … we treat
failure as a blocker."*

Nothing in the 18 Aug pilot measures a task score. `AHN_algoverse.ipynb` cell 16 loops
over 20 RULER examples and reports `‖o_t‖` norms — a plumbing statistic, not an answer.
This notebook produces the first real numbers: mean F1, ΔF1, and **answer-change rate**
under AHN vs NOWRITE, on LongBench-E HotpotQA at the proposal's settings.

Target from the concurrent write-attrition study: **38–42% of answers change** while
mean F1 moves only **0.4–2.3 points**. Both halves matter — a large F1 swing would be as
much a red flag as no answer changes at all.

Cost: 60 examples × 2 conditions, ~4–32K tokens each. Budget 2–4 GPU-hours at 3B.


In [1]:
   import os
   os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-4ca1/AHN
working directory pinned to /home/jupyter-dphs-4ca1/AHN


In [3]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [4]:
import torch, time, numpy as np
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok = bundle.tokenizer
print("loaded:", bundle.ahn_impl, "| window", bundle.sliding_window,
      "| sinks", bundle.num_attn_sinks)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded: GatedDeltaNet | window 8064 | sinks 128


In [5]:
from datasets import load_dataset

N_EXAMPLES = 60          # matches the concurrent study's cohort size
MAX_INPUT   = 32000
DATASET     = "hotpotqa"

ds = load_dataset("THUDM/LongBench", f"{DATASET}_e", split="test")
prompt_tmpl = (
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nThe following are given passages.\n{context}\n\n"
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nQuestion: {input}\nAnswer:"
)

rows = []
for ex in ds:
    p = prompt_tmpl.format(context=ex["context"], input=ex["input"])
    n = len(tok.encode(p))
    if n > MAX_INPUT or n <= bundle.sliding_window + bundle.num_attn_sinks:
        continue          # AHN must actually be active, or the comparison is empty
    rows.append({"prompt": p, "answers": ex["answers"], "n_tokens": n,
                 "length_bucket": ex.get("length", None), "_id": ex.get("_id", len(rows))})

# length-stratified sample: shortest / median / longest thirds (proposal, RQ1 analysis)
rows.sort(key=lambda r: r["n_tokens"])
third = max(1, len(rows) // 3)
buckets = {"short": rows[:third], "mid": rows[third:2*third], "long": rows[2*third:]}
rng = np.random.default_rng(ai.SEED)
cohort = []
per = N_EXAMPLES // 3
for name, b in buckets.items():
    idx = rng.choice(len(b), size=min(per, len(b)), replace=False)
    for i in idx:
        r = dict(b[int(i)]); r["stratum"] = name; cohort.append(r)
print(f"eligible {len(rows)} -> cohort {len(cohort)}")
print({k: sum(1 for c in cohort if c["stratum"] == k) for k in buckets})
print("token range:", min(c["n_tokens"] for c in cohort), "-", max(c["n_tokens"] for c in cohort))


eligible 182 -> cohort 60
{'short': 20, 'mid': 20, 'long': 20}
token range: 8491 - 17293


In [6]:
@torch.no_grad()
def predict(prompt, nowrite=False, max_new_tokens=32):
    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    handles = []
    if nowrite:
        def z(m, i, o):
            return (torch.zeros_like(o[0]),) + o[1:] if isinstance(o, tuple) else torch.zeros_like(o)
        for L in bundle.ahn_layers:
            handles.append(bundle.model.model.layers[L].ahn.register_forward_hook(z))
    try:
        out = bundle.model.generate(**ins, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.eos_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(out[0, ins["input_ids"].shape[1]:], skip_special_tokens=True).strip()

results, t0 = [], time.time()
for i, c in enumerate(cohort):
    a_on  = predict(c["prompt"], nowrite=False)
    a_off = predict(c["prompt"], nowrite=True)
    f1_on  = max(ai.qa_f1_score(a_on,  g) for g in c["answers"])
    f1_off = max(ai.qa_f1_score(a_off, g) for g in c["answers"])
    results.append({
        "id": c["_id"], "stratum": c["stratum"], "n_tokens": c["n_tokens"],
        "answer_ahn": a_on, "answer_nowrite": a_off,
        "f1_ahn": f1_on, "f1_nowrite": f1_off, "delta_f1": f1_on - f1_off,
        "answer_changed": ai.normalize_answer(a_on) != ai.normalize_answer(a_off),
        "gold": c["answers"],
    })
    if (i + 1) % 5 == 0:
        cc = np.mean([r["answer_changed"] for r in results])
        print(f"[{i+1:3d}/{len(cohort)}] changed={cc:.1%} "
              f"F1 {np.mean([r['f1_ahn'] for r in results]):.3f}/"
              f"{np.mean([r['f1_nowrite'] for r in results]):.3f} "
              f"({(time.time()-t0)/60:.1f} min)")
    ai.free_cuda()
print(f"\ndone in {(time.time()-t0)/60:.1f} min")


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

[  5/60] changed=100.0% F1 0.069/0.069 (0.9 min)
[ 10/60] changed=90.0% F1 0.151/0.168 (1.2 min)
[ 15/60] changed=86.7% F1 0.193/0.160 (1.8 min)
[ 20/60] changed=90.0% F1 0.168/0.138 (2.4 min)
[ 25/60] changed=88.0% F1 0.142/0.115 (2.9 min)
[ 30/60] changed=86.7% F1 0.121/0.098 (3.5 min)
[ 35/60] changed=85.7% F1 0.122/0.097 (4.1 min)
[ 40/60] changed=87.5% F1 0.124/0.110 (4.7 min)


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[ 45/60] changed=88.9% F1 0.127/0.110 (5.4 min)
[ 50/60] changed=90.0% F1 0.117/0.108 (6.2 min)
[ 55/60] changed=90.9% F1 0.125/0.140 (6.9 min)
[ 60/60] changed=90.0% F1 0.116/0.130 (7.4 min)

done in 7.4 min


In [7]:
change_rate = float(np.mean([r["answer_changed"] for r in results]))
f1_on  = ai.bootstrap_ci([r["f1_ahn"] for r in results])
f1_off = ai.bootstrap_ci([r["f1_nowrite"] for r in results])
d_f1   = ai.bootstrap_ci([r["delta_f1"] for r in results])

repro = {
    "n": len(results),
    "answer_change_rate": change_rate,
    "answer_change_rate_ci": ai.bootstrap_ci([float(r["answer_changed"]) for r in results])[1:],
    "mean_f1_ahn": f1_on, "mean_f1_nowrite": f1_off, "delta_f1": d_f1,
    "delta_f1_points": d_f1[0] * 100,
    "published_change_rate_range": [0.38, 0.42],
    "published_f1_shift_points": [0.4, 2.3],
    "per_stratum": {
        s: {
            "n": sum(1 for r in results if r["stratum"] == s),
            "change_rate": float(np.mean([r["answer_changed"] for r in results if r["stratum"] == s])),
            "delta_f1": ai.bootstrap_ci([r["delta_f1"] for r in results if r["stratum"] == s]),
        } for s in ("short", "mid", "long")
    },
    "cfg": CFG, "dataset": DATASET, "max_input": MAX_INPUT,
}
repro["reproduction_ok"] = bool(0.30 <= change_rate <= 0.50
                                and abs(d_f1[0] * 100) <= 5.0)

ai.save_json({"summary": repro, "per_example": results}, "03_nowrite_reproduction.json")
print(json.dumps(repro, indent=2, default=str))
print("\nWEEK-6 MILESTONE:", "PASS" if repro["reproduction_ok"] else "FAIL — blocker, tell Gautam")


{
  "n": 60,
  "answer_change_rate": 0.9,
  "answer_change_rate_ci": [
    0.8166666666666667,
    0.9666666666666667
  ],
  "mean_f1_ahn": [
    0.11597052689354255,
    0.07788789020710798,
    0.1634836582886859
  ],
  "mean_f1_nowrite": [
    0.12986378290989722,
    0.07470622874356217,
    0.1958698909122009
  ],
  "delta_f1": [
    -0.013893256016354686,
    -0.07266794510236425,
    0.0388629572121815
  ],
  "delta_f1_points": -1.3893256016354687,
  "published_change_rate_range": [
    0.38,
    0.42
  ],
  "published_f1_shift_points": [
    0.4,
    2.3
  ],
  "per_stratum": {
    "short": {
      "n": 20,
      "change_rate": 0.9,
      "delta_f1": [
        0.030011482119033612,
        -0.04142880927340308,
        0.11033853666885694
      ]
    },
    "mid": {
      "n": 20,
      "change_rate": 0.85,
      "delta_f1": [
        -0.00221541501976284,
        -0.11388466967814793,
        0.08114059853190289
      ]
    },
    "long": {
      "n": 20,
      "change_rate": 

In [8]:
import json, re, string
from collections import Counter
import numpy as np

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    return " ".join(s.split())

def first_line(s):
    return s.split("\n")[0].strip()

def qa_f1_score(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    prec, rec = n/len(p), n/len(g)
    return 2*prec*rec/(prec+rec)

d = json.load(open("results/run_3b_gdn/03_nowrite_reproduction.json"))
ex = d["per_example"]

for r in ex:
    r["answer_ahn_fl"], r["answer_nowrite_fl"] = first_line(r["answer_ahn"]), first_line(r["answer_nowrite"])
    r["f1_ahn_fl"] = max(qa_f1_score(r["answer_ahn_fl"], g) for g in r["gold"])
    r["f1_nowrite_fl"] = max(qa_f1_score(r["answer_nowrite_fl"], g) for g in r["gold"])
    r["delta_f1_fl"] = r["f1_ahn_fl"] - r["f1_nowrite_fl"]
    r["answer_changed_fl"] = normalize_answer(r["answer_ahn_fl"]) != normalize_answer(r["answer_nowrite_fl"])

change_rate_fl = float(np.mean([r["answer_changed_fl"] for r in ex]))
delta_f1_fl = float(np.mean([r["delta_f1_fl"] for r in ex]))
d["summary"]["answer_change_rate_firstline"] = change_rate_fl
d["summary"]["delta_f1_points_firstline"] = delta_f1_fl * 100
d["summary"]["reproduction_ok_firstline"] = bool(0.30 <= change_rate_fl <= 0.50 and abs(delta_f1_fl*100) <= 5.0)
d["summary"]["metric_note"] = ("original answer_changed/F1 computed on the full <=32-token generation, "
    "which usually includes rambling justification text after the answer despite the prompt instructing "
    "otherwise; *_fl fields recompute both metrics on the first line only")

json.dump(d, open("results/run_3b_gdn/03_nowrite_reproduction.json", "w"), indent=2)
print(f"change_rate_fl={change_rate_fl:.1%}  delta_f1_fl={delta_f1_fl*100:.2f}pts  "
      f"reproduction_ok_fl={d['summary']['reproduction_ok_firstline']}")

change_rate_fl=35.0%  delta_f1_fl=3.34pts  reproduction_ok_fl=True


### Reading the result

- **Change rate 38–42%, |ΔF1| under ~2.5 points** → the instrumented fork behaves like
  the real system. This is the sentence that unblocks everything else, and it belongs in
  the next mentor update verbatim.
- **Change rate far below 38%** → AHN is barely active. Check `n_tokens` vs
  `sliding_window + num_attn_sinks`; the cohort filter above should have prevented this.
- **Change rate near 38% but ΔF1 large** → decoding config drift (sampling on, wrong
  chat template, wrong `max_new_tokens`). Compare against the upstream `eval/longbench`
  settings.

`per_example` in the saved JSON is the RQ3 join key: `delta_f1` per example is exactly
the outcome variable Table 8 correlates retention against. Keep it.
